# MegaAI · M2 — `screen-classifier`

Trains the model that answers **“what screen am I looking at?”** — login, signup, checkout,
cart, product-list, dashboard, contact-form, article, error, settings.

Desktop automation uses this to know where it is before it clicks anything.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

**Output:** `screen-classifier.pt`, `.onnx` and a labels file.


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — in Colab use Runtime -> Change runtime type -> T4 GPU (training will be slow otherwise).")


In [ ]:
!pip install -q torch torchvision onnx onnxruntime pyyaml


## 1. Load the dataset


In [ ]:
# Upload the dataset.zip produced by:
#   node scripts/dataset/generate-ui-dataset.mjs --count 3000 --out dataset
#   zip -r dataset.zip dataset
import os, zipfile

if not os.path.exists("dataset"):
    try:
        from google.colab import files
        print("Choose your dataset.zip …")
        up = files.upload()
        name = list(up.keys())[0]
    except Exception:
        name = "dataset.zip"          # running locally: put dataset.zip beside the notebook
    with zipfile.ZipFile(name) as z:
        z.extractall(".")

# The zip may contain dataset/ at the root or one level down — find data.yaml.
root = None
for base, dirs, fs in os.walk("."):
    if "data.yaml" in fs and "images" in dirs:
        root = os.path.abspath(base)
        break
assert root, "Could not find the dataset (no data.yaml with an images/ folder)"

import yaml
_names = yaml.safe_load(open(os.path.join(root, "data.yaml")))["names"]
CLASSES = [_names[i] for i in sorted(_names)] if isinstance(_names, dict) else list(_names)

print("dataset root:", root)
print("classes     :", CLASSES)
print("train images:", len(os.listdir(os.path.join(root, "images/train"))))
print("val images:  ", len(os.listdir(os.path.join(root, "images/val"))))


## 2. Train


In [ ]:
import os, csv, random
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image

CSV = os.path.join(root, "screens.csv")
rows = list(csv.DictReader(open(CSV)))
LABELS = sorted({r["kind"] for r in rows})
label_to_idx = {l: i for i, l in enumerate(LABELS)}
print(len(rows), "images |", len(LABELS), "classes:", LABELS)

IMG = 224
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ScreenDataset(Dataset):
    def __init__(self, split, tf):
        self.rows = [r for r in rows if r["split"] == split]
        self.tf = tf
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        path = os.path.join(root, "images", r["split"], r["file"])
        return self.tf(Image.open(path).convert("RGB")), label_to_idx[r["kind"]]

train_ds, val_ds = ScreenDataset("train", train_tf), ScreenDataset("val", eval_tf)
print("train", len(train_ds), "| val", len(val_ds))
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=32, num_workers=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(LABELS))
model = model.to(device)

EPOCHS = 12
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
    sched.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.numel()
    acc = correct / max(1, total)
    if acc > best_acc:
        best_acc = acc
        torch.save({"state_dict": model.state_dict(), "labels": LABELS}, "screen-classifier.pt")
    print(f"epoch {epoch+1:2d}/{EPOCHS}  val accuracy {acc*100:5.1f}%")

print(f"\nbest held-out accuracy: {best_acc*100:.1f}%  ->  screen-classifier.pt")


## 3. Per-class results on the held-out split


In [ ]:
from collections import defaultdict
import torch

ckpt = torch.load("screen-classifier.pt", map_location=device)
model.load_state_dict(ckpt["state_dict"]); model.eval()

hits, totals = defaultdict(int), defaultdict(int)
with torch.no_grad():
    for x, y in val_dl:
        pred = model(x.to(device)).argmax(1).cpu()
        for p, t in zip(pred, y):
            totals[LABELS[t]] += 1
            if p == t: hits[LABELS[t]] += 1

print("per-class accuracy on the held-out split")
for label in LABELS:
    n = totals[label]
    print(f"  {label:16s} {(hits[label]/n*100 if n else 0):5.1f}%   ({hits[label]}/{n})")


## 4. Export and download


In [ ]:
import torch
dummy = torch.randn(1, 3, IMG, IMG, device=device)
torch.onnx.export(model, dummy, "screen-classifier.onnx",
                  input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=12)

import json
json.dump({"labels": LABELS, "imgSize": IMG}, open("screen-classifier.labels.json", "w"), indent=2)
print("exported screen-classifier.pt / .onnx / .labels.json")

try:
    from google.colab import files
    files.download("screen-classifier.pt")
    files.download("screen-classifier.onnx")
    files.download("screen-classifier.labels.json")
except Exception as e:
    print("Not on Colab — files are in the working directory.", e)
